# Notebook16 shared source and training-data helpers

Purpose: load tagged definitions, verify frozen sources/checkpoints, preserve original Food101 membership and configure deterministic GPU execution. Inputs: Notebook15 handoff or declared Notebook14 ViT fallback. Outputs: source-bound model/data helpers; fallback never supplies a quality claim. Definitions do not launch anything. Runtime seconds plus image-hash verification. Checkpoints remain under the active16run; official-test pixels are never read here.

In [ ]:
"""Shared source/data/checkpoint helpers for the gated16 workflow; definitions never train."""
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
from pathlib import Path
import copy,hashlib,json,sys,types,time
import nbformat,torch
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()

def export_hash(path):
    return hashlib.sha256('\n'.join(c.source for c in nbformat.read(path,4).cells if c.cell_type=='code'and'ahr-export'in c.metadata.get('tags',[])).encode()).hexdigest()

def load(root,relative,name):
    root=Path(root).resolve();path=root/relative
    if name in sys.modules:
        module=sys.modules[name]
        if Path(module.__file__).resolve()!=path.resolve():raise ValueError('Module path changed')
        return module
    if not path.exists():raise FileNotFoundError('Required tagged notebook: '+str(path))
    module=types.ModuleType(name);module.__file__=str(path);sys.modules[name]=module
    selected=[c for c in nbformat.read(path,4).cells if c.cell_type=='code'and'ahr-export'in c.metadata.get('tags',[])]
    if not selected:raise ValueError('No tagged reusable definitions')
    try:
        for index,cell in enumerate(selected):exec(compile(cell.source,str(path)+'#ahr-export-'+str(index),'exec'),module.__dict__)
    except BaseException:sys.modules.pop(name,None);raise
    return module

def bootstrap(root):
    module=load(root,'fmm_jepa/bootstrap.ipynb','fmm16_bootstrap');module.install_notebook_package(Path(root));return module

def modules(root):
    bootstrap(root)
    runner=load(root,'experiments/15_vit_geometry/runner.ipynb','fmm15_runner')
    encoder=load(root,'experiments/15_vit_geometry/encoder.ipynb','fmm15_encoder')
    operator=load(root,'experiments/16_hierarchical_confirmation/operator.ipynb','fmm16_operator')
    return runner,encoder,operator

def source_hashes(root):
    root=Path(root).resolve();runner,_,_=modules(root);result=runner.stage_definition_hashes(root)
    for path in sorted((root/'experiments/16_hierarchical_confirmation').glob('*.ipynb')):result[str(path.relative_to(root))]=export_hash(path)
    return result

def canonical(value):return json.loads(json.dumps(value,allow_nan=False))
def digest(value):return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def file_record(path):
    path=Path(path);return {'path':str(path.resolve()),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
def atomic_json(path,value):
    bootstrap(ROOT);from fmm_jepa.training import atomic_json as write;return write(path,canonical(value))
def read(path):return json.loads(Path(path).read_text())
def immutable(path,value):
    value=canonical(value);path=Path(path)
    if path.exists():
        if read(path)!=value:raise ValueError('Immutable registration changed:'+str(path))
    else:atomic_json(path,value)
    return value

def check_deadline(deadline,required=0.):
    if deadline is not None and time.time()+float(required)>=float(deadline):raise TimeoutError('Original16deadline insufficient/expired; UNKNOWN, never reset')

def environment(config):
    if os.environ.get('CUBLAS_WORKSPACE_CONFIG')!=':4096:8':raise RuntimeError('Deterministic CUDA workspace required')
    torch.set_num_threads(2);torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    device=str(config.get('device','cuda:0'))
    if device.startswith('cuda'):
        if not torch.cuda.is_available():raise RuntimeError('Registered GPU unavailable')
        torch.cuda.set_device(torch.device(device))
    return {'device':device,'dtype':'float32','tf32':False,'deterministic':True,'cpu_threads':2}

def data_bundle(root,artifact,config,verify_images=True):
    root,artifact=Path(root),Path(artifact);bootstrap(root)
    data=load(root,'experiments/05_scaling/preparation.ipynb','fmm16_data05')
    def paths(source,folder,stage):
        folder=Path(folder).resolve();relative=folder.relative_to(root/'artifacts/notebook_16');run_id=relative.parts[0]
        return folder,root/'checkpoints'/run_id/'data',run_id
    data._stage_paths=paths
    stage_id='resolution'if int(config['image_size'])==256 else'learning'
    if int(config['image_size'])not in(128,256):raise ValueError('Registered16data resolutions128or256 only')
    return data.prepare_stage_bundle(root,artifact,{'id':stage_id,'image_size':int(config['image_size']),'pretrain_pool':'selected'},config,verify_images=verify_images)

def handoff(root,source_run=None):
    folder=Path(root)/'artifacts/notebook_15'
    candidates=[folder/source_run/'handoff_15_to_16.json']if source_run else sorted(folder.glob('*/handoff_15_to_16.json'),key=lambda p:p.stat().st_mtime,reverse=True)
    for path in candidates:
        if path.exists():
            value=read(path)
            if not source_run and (value.get('fixture_source') or value.get('status')=='VALIDATION_ONLY'):continue
            return {**value,'handoff_record':file_record(path)}
    return {'status':'NOT_AVAILABLE','candidate_variant':'optional_geometry','quality_gate':False,'reason':'Notebook15 scientific handoff is not complete. Operator benchmarks use frozen14ViT training-only fallback; confirmation stays closed.'}

def _path(value):return value.get('path')if isinstance(value,dict)else value

def learned_source(root,handoff_value,variant='optional_geometry'):
    """Predeclared first registered seed only; never choose the best-scoring seed."""
    root=Path(root);runner,encoder,operator=modules(root)
    plan_path=next((_path(handoff_value[k])for k in('candidate_plan','plan_path','plan','source_plan')if handoff_value.get(k)),None)
    training_path=next((_path(handoff_value[k])for k in('training_result','training_path','training')if handoff_value.get(k)),None)
    if handoff_value.get('status')=='COMPLETE'and plan_path and training_path:
        plan=read(plan_path);training=read(training_path);seed=sorted(plan['seeds'])[0]
        arm=next(a for a in plan['arms']if int(a['seed'])==seed and a.get('variant')==variant)
        config=copy.deepcopy(arm['config']);record=training['arms'][arm['id']]['milestones'][str(plan['updates'])]
        payload=torch.load(record['path'],map_location='cpu',weights_only=False)
        if file_record(record['path'])['sha256']!=record['sha256']:raise ValueError('Frozen15teacher checksum mismatch')
        model=runner._default_build_model(config,arm['kind']).cpu();model.load_state_dict(payload['model'])
        return model,config,{'role':'FROZEN_NOTEBOOK15_TRAINING_CHECKPOINT','seed':seed,'variant':variant,'checkpoint':record,'plan':file_record(plan_path),'training':file_record(training_path)}
    old=root/'artifacts/notebook_14/routing-capacity-001';plan=read(old/'plan.json');arm=next(a for a in plan['arms']if a['id']=='s20261601-vit')
    config=copy.deepcopy(arm['config']);config.pop('kernel14',None)
    config['kernel15']={'variant':variant,'spatial_coefficient_bound':2.,'spatial_length':.35,'temperature_trainable':False,'spatial_length_trainable':False,'spatial_coefficient_trainable':True}
    training=read(old/'workers/worker0/training_result.json');record=training['arms'][arm['id']]['milestones']['600']
    if file_record(record['path'])['sha256']!=record['sha256']:raise ValueError('Fallback checkpoint checksum mismatch')
    payload=torch.load(record['path'],map_location='cpu',weights_only=False);model=runner._default_build_model(config,'semantic'if variant!='vit'else'vit').cpu()
    incompatible=model.load_state_dict(payload['model'],strict=False)
    if incompatible.unexpected_keys or any(not any(k.endswith(s)for s in('log_temperature_multiplier','spatial_coefficient_raw','log_spatial_length'))for k in incompatible.missing_keys):raise ValueError('Fallback rawViT state differs beyond declared zero geometry parameters')
    return model,config,{'role':'FROZEN_NOTEBOOK14_TRAINING_ONLY_FALLBACK_NO15QUALITY','seed':20261601,'variant':variant,'checkpoint':record,'geometry_initially_zero':True,'official_test':'NOT_READ'}

def builder(root,hierarchy_config):
    runner,encoder,operator=modules(root)
    def build(config,kind):
        model=runner._default_build_model(config,kind)
        if kind=='semantic':model=operator.replace_encoder_mixers(model,dict(hierarchy_config))
        return model
    return build

def bind_backend(root,hierarchy_config):
    runner,encoder,operator=modules(root);identity=source_hashes(root);runner.bind_model_builder(builder(root,hierarchy_config),identity)
    return runner,identity
